# LUTM-1: temperature-like island GA

This is the research trainer for delta-dependent tournament selection, neutral elite turnover, segment crossover, migration, and random immigrants. NumPy evolves the islands; Taichi simulates the complete population on CUDA.

Run this notebook with **`LUTM-1` as the working directory** and the `slackenv` kernel. Training progress is printed by the trainer while it runs.

In [ ]:
from pathlib import Path

required_files = ("lutm.py", "taichi_backend.py", "island_ga.py", "utils.py")
missing = [name for name in required_files if not (Path.cwd() / name).is_file()]
if missing:
    raise RuntimeError(
        "Run this notebook from the LUTM-1 repository root; missing: "
        + ", ".join(missing)
    )

from island_ga import IslandGAConfig, TaichiIslandGA
from taichi_backend import (
    TaichiUTMSimulator,
    evaluate_program_taichi,
    initialize_taichi_cuda,
)
from utils import SimulatorConfig, TaskCases

initialize_taichi_cuda()

## Target task

Edit the two lists directly. They must have equal length, inputs must be unique, and every value must be binary. Inputs may include `""`; targets must be nonempty.

In [ ]:
inputs = ["00", "11", "01", "10", "1", "0"]
targets = ["0000", "1111", "0101", "1010", "11", "00"]

task = TaskCases(inputs, targets)
for input_bits, target in task.pairs():
    print(f"{input_bits!r:>6} -> {target!r}")

## Simulator and search settings

Fitness is normalized positional bit accuracy minus `k_penalty` times the invalid fraction. Each parent is chosen by an independent two-member tournament. With the settings below, the fitter contestant is selected with probability 0.53 for `0 < delta <= 0.05`, 0.58 for `0.05 < delta <= 0.20`, and 0.80 above 0.20; exact ties are sampled 50/50.

Crossover is applied independently to each child and copies three uniformly sampled segments from parent 2 while retaining parent 1's effective length. New children are preferred over carried elites when fitness ties exactly. Migration occurs at the selected interval; random immigrants are introduced every generation.

In [ ]:
simulation_config = SimulatorConfig(
    program_width=400,
    left_budget=410,
    right_budget=30,
    t_max=48_000,
)

ga_config = IslandGAConfig(
    islands=200,
    population_per_island=800,
    generations=16_000,
    elite_fraction=0.03,
    cross_island_fraction=0.02,
    cross_island_interval=1_200,
    random_immigrant_fraction=0.15,
    tournament_delta_thresholds=(0.05, 0.20),
    tournament_best_probabilities=(0.53, 0.58),
    tournament_base_probability=0.80,
    p_crossover=0.50,
    crossover_segments=3,
    crossover_segment_min=1,
    crossover_segment_max=20,
    p_insert=0.04,
    p_delete=0.06,
    p_bit_flip=0.02,
    k_penalty=0.5,
    seed=4,
    print_every=20,
    stop_first_exact=True,
)

print(f"total CUDA population:              {ga_config.total_population:,}")
print(f"elites per island:                 {ga_config.elite_count:,}")
print(f"scheduled migrants per island:     {ga_config.cross_island_count:,}")
print(f"random immigrants per island:      {ga_config.random_immigrant_count:,}")

## Train

This cell allocates the CUDA simulator and starts the search. The trainer prints the best fitness, mean island-best fitness, accuracy, invalid fraction, exact-match count, promoted neutral children, crossover count, best program, and elapsed time every `print_every` generations.

In [ ]:
simulator = TaichiUTMSimulator(
    simulation_config,
    batch_capacity=ga_config.total_population,
)
trainer = TaichiIslandGA(simulator, task, ga_config)
result = trainer.run()

print()
print(f"solved:                 {result.solved}")
print(f"generations completed:  {result.generations_completed:,}")
print(f"program evaluations:    {result.evaluated_programs:,}")
print(f"elapsed:                {result.elapsed_seconds:.3f} s")
print(f"best generation:        {result.best.generation:,}")
print(f"best island:            {result.best.island + 1:,}")
print(f"best fitness:           {result.best.fitness:.6f}")
print(f"best bit accuracy:      {result.best.bit_accuracy:.6f}")
print(f"best invalid fraction:  {result.best.invalid_fraction:.6f}")
print(f"best program length:    {result.best.effective_length:,}")
print(f"best program:           {result.best.program!r}")
print(f"padded program:         {result.best.padded_program!r}")

## Verify the best-ever program

The best program is archived in memory even if its island later moves elsewhere. Run it again on every task case and inspect its decoded output and physical computation diagnostics.

In [ ]:
evaluations = evaluate_program_taichi(simulator, result.best.program, task)

print(f"{'input':>8}  {'target':>8}  {'output':>8}  {'exact':>5}  {'reason':>22}  {'T':>10}  {'L/R space':>16}")
for case in evaluations:
    print(
        f"{case.input_bits!r:>8}  {case.target!r:>8}  {case.output!r:>8}  "
        f"{str(case.exact):>5}  {case.invalid_reason:>22}  {case.T:>10,}  "
        f"{case.left_space_used:>7,}/{case.right_space_used:<7,}"
    )

## Inspect training history

The complete generation-by-generation history remains in memory. The cell below prints the final twenty generations and the first twenty final island champions.

In [ ]:
history = result.history
tail = min(20, history.generations.size)
start = history.generations.size - tail

print(f"{'generation':>10}  {'global best':>11}  {'mean fitness':>12}  {'accuracy':>8}  {'invalid':>8}  {'exact':>7}")
for index in range(start, history.generations.size):
    print(
        f"{int(history.generations[index]):>10,}  "
        f"{history.global_best_fitness[index]:>11.6f}  "
        f"{history.mean_fitness[index]:>12.6f}  "
        f"{history.best_bit_accuracy[index]:>8.6f}  "
        f"{history.best_invalid_fraction[index]:>8.6f}  "
        f"{int(history.exact_programs[index]):>7,}"
    )

shown_islands = min(20, ga_config.islands)
print("\nFinal best fitness by island (first 20):")
for island, fitness in enumerate(history.island_best_fitness[-1, :shown_islands], start=1):
    print(f"island {island:>3}: {fitness:.6f}")